In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import json

In [3]:
# input_data(페르소나)
input_data1 = """\
저는 취준생이라 소득이 적어요. 매일 지하철+버스 타고 학원 다니고, 밥은 편의점에서 해결합니다. 연회비 부담 없고 대중교통이랑 편의점 할인 되는 카드 추천해주세요.\
"""

input_data2 = """\
30대 직장인입니다. 온라인 쇼핑이랑 카페를 많이 이용하고 해외여행도 자주 갑니다. 해외결제 수수료 낮고 쇼핑 할인 되는 카드 추천해주세요. 연회비 3만원까지 괜찮아요.\
"""

input_data3 = """\
프리랜서라 월 소득이 불규칙해요. 전월실적 조건 없는 카드가 제일 중요합니다. 카페를 자주 이용하고 대중교통도 많이 타요.\
"""

input_data4 = """\
맞벌이 신혼부부입니다. 주유비, 마트 장보기, 공과금 할인이 많이 되는 카드를 찾고 있어요. 연회비는 5만원까지 괜찮습니다. 롯데카드는 빼주세요.\
"""

input_data5 = """\
60대 은퇴한 사람입니다. 병원이랑 약국을 자주 가는데 할인 되는 카드 추천해주세요. 마트 할인도 되면 좋겠어요. 복잡한 건 싫고 연회비 저렴한 카드면 좋겠습니다.\
"""

# 시스템 프롬프트 정의
system_prompt = """\
역할: 당신은 “한국 신용카드 추천 챗봇”이다. 반드시 유저 프롬프트에 포함된 정보에 기반해 카드 추천을 한 번에 제공한다. 카드 정보는 카드고릴라 사이트를 확인한다.

절대 규칙(위반 금지)
1) 질문 금지: 어떤 경우에도 질문/역질문/확인 요청을 하지 않는다.
2) 근거 제한: 추천 판단은 “유저 프롬프트에 명시된 내용”만 사용한다.
   - 유저가 명시한 선호/제약(연회비 상한, 혜택 카테고리, 카드사 선호/비선호, 해외/국내, 할인/적립/마일, 실적 부담 등)이 있으면 최우선 반영한다.
   - 유저 프롬프트가 모호하거나 정보가 부족하면, “정보 부족 모드”로 처리하고 범용 포트폴리오(4유형 커버리지)로 추천한다.
3) 출력 포맷 강제: 출력은 카드 블록 + 구분선만 허용한다. 다른 설명/서론/요약/면책/추가 안내 금지.
4) 카드 수: 반드시 4개만 출력한다(초과/미만 금지).
5) 최신성/단종: 불확실하면 카드혜택에 “발급 가능 여부 확인 필요”라고만 쓴다.
6) 추천이유 규칙: 추천이유는 5줄 이내(줄바꿈 기준). 질문 포함 금지. 과장(무조건/최고/절대) 금지.
7) 문체: 존댓말을 기본으로 예의있고 상냥한 말투로 답변한다.

정보 부족 모드(유저 정보가 거의 없을 때)의 내부 선택 로직
- 대표 소비 패턴 4유형을 커버하도록 1개씩 선택한다:
  A) 조건 단순/관리 부담 낮음(범용 할인/적립)
  B) 고정지출(통신/공과금/생활비)
  C) 트렌드/구독/외식·배달
  D) 여행/마일리지/해외결제
- 선택 우선순위: 접근성(연회비/조건 과도X) > 범용성 > 컨셉 명확성 > 검증성
- 단, 유저 프롬프트에 특정 카테고리/제약이 있으면 해당 축의 가중치를 가장 높게 둔다.

상세페이지 규칙
- “상세페이지”에는 카드고릴라 사이트에 카드상세페이지 URL을 우선 기입한다.
- 확정 불가 시 “확인 필요”라고만 쓴다.

출력 형식(정확히 이 구조 + 카드 사이 구분선 필수)
[카드 1]
카드사: ...
카드이름: ...
연회비: ...
카드혜택: ...
상세페이지: ...
추천이유: ...
(추천이유는 최대 5줄)
====================
[카드 2]
카드사: ...
카드이름: ...
연회비: ...
카드혜택: ...
상세페이지: ...
추천이유: ...
(추천이유는 최대 5줄)
====================
[카드 3]
카드사: ...
카드이름: ...
연회비: ...
카드혜택: ...
상세페이지: ...
추천이유: ...
(추천이유는 최대 5줄)
====================
[카드 4]
카드사: ...
카드이름: ...
연회비: ...
카드혜택: ...
상세페이지: ...
추천이유: ...
(추천이유는 최대 5줄)\
"""

# 1. Base모델

In [5]:
# 모델정의
model = "gpt-5-nano"
model = ChatOpenAI(model=model, temperature=0.8)

# 유저 프롬프트 정의
user_prompt = """\
아래 사용자의 정보를 보고 그 상황에 맞는 알맞은 카드를 추천해주세요.

[사용자 정보]
{input_data}\
"""

# 템플릿 생성
template = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", user_prompt)
])


# 체인 생성
base_chain1 = template | model | StrOutputParser()
base_chain2 = template | model | StrOutputParser()
base_chain3 = template | model | StrOutputParser()
base_chain4 = template | model | StrOutputParser()
base_chain5 = template | model | StrOutputParser()

print("첫번째 고객\n", input_data1)
print()
print("답변:\n", base_chain1.invoke(input_data1))
print()
print("두번째 고객\n", input_data2)
print()
print("답변:\n", base_chain2.invoke(input_data2))
print()
print("세번째 고객\n", input_data3)
print()
print("답변:\n", base_chain3.invoke(input_data3))
print()
print("네번째 고객\n", input_data4)
print()
print("답변:\n", base_chain4.invoke(input_data4))
print()
print("다섯번째 고객\n", input_data5)
print()
print("답변:\n", base_chain5.invoke(input_data5))

첫번째 고객
 저는 취준생이라 소득이 적어요. 매일 지하철+버스 타고 학원 다니고, 밥은 편의점에서 해결합니다. 연회비 부담 없고 대중교통이랑 편의점 할인 되는 카드 추천해주세요.

답변:
 죄송하지만 현재 이 대화에서 카드고릴라의 최신 상세 페이지를 확인할 수 없어 정확한 4장 추천을 바로 드리기 어렵습니다. 원하시면 제가 카드고릴라 데이터를 확인한 뒤, 요청하신 조건(연회비 0원, 대중교통 및 편의점 할인)을 충족하는 4장으로 구성해 드리겠습니다.

두번째 고객
 30대 직장인입니다. 온라인 쇼핑이랑 카페를 많이 이용하고 해외여행도 자주 갑니다. 해외결제 수수료 낮고 쇼핑 할인 되는 카드 추천해주세요. 연회비 3만원까지 괜찮아요.

답변:
 [카드 1]
카드사: 현대카드
카드이름: 현대카드 ZERO Travel
연회비: 0원
카드혜택: 해외결제 수수료 면제(0%), 온라인 쇼핑 할인, 카페 제휴 할인
상세페이지: 확인 필요
추천이유: 해외여행 시 수수료 부담을 낮추고 온라인 쇼핑 할인도 기대할 수 있습니다.

[카드 2]
카드사: 삼성카드
카드이름: 삼성카드 2 V
연회비: 15,000원
카드혜택: 해외결제 수수료 저렴, 온라인 쇼핑 할인, 카페 제휴 할인
상세페이지: 확인 필요
추천이유: 해외 결제 수수료를 실질적으로 절감하면서 온라인 쇼핑과 카페 이용 혜택을 함께 누릴 수 있습니다.

[카드 3]
카드사: 신한카드
카드이름: 신한카드 The Style 온라인
연회비: 12,000원
카드혜택: 온라인 쇼핑 할인(다양한 카테고리), 해외결제 수수료 낮음, 제휴 할인
상세페이지: 확인 필요
추천이유: 온라인 쇼핑 중심의 사용 패턴과 해외 결제 혜택을 균형 있게 제공합니다.

[카드 4]
카드사: KB국민카드
카드이름: 국민카드 옥션 Travel
연회비: 9,900원
카드혜택: 해외결제 수수료 낮음, 온라인 쇼핑 할인, 카페/외식 할인
상세페이지: 확인 필요
추천이유: 합리적 연회비로 해외 결제와 온라인 쇼핑 혜택을 함께 활용하기에 적합합니다.

세번째 고객


# 2. FewShot 모델

In [12]:
# FewShot 데이터 정의
examples = [
    {
        "User_Input": """
저는 20대 취준생입니다. 매일 대중교통으로 학원에 가고, 식비는 주로 편의점에서 해결해요 전월실적 부담이 적고 대중교통, 편의점 할인이 되는 카드를 원합니다.
        """,
        "Output":"""
[카드 1]
카드사: 신한카드
카드이름: 신한카드 Mr.Life
연회비: 해외겸용 15,000 원
카드혜택: 편의점 10% 할인, 병원/약국 10% 할인, 공과금 10% 할인
상세페이지: https://www.card-gorilla.com/card/detail/13
추천이유: 편의점 결제 시 10% 할인을 제공하여 식비 절감에 유리해 추천했습니다. 공과금과 통신비 할인도 포함되어 자취하는 취업준비생의 고정지출 방어에 적합합니다. 전월실적 30만 원으로 비교적 유지 부담이 적은 편입니다.
====================
[카드 2]
카드사: 삼성카드
카드이름: 삼성카드 taptap O
연회비: 국내전용 10,000 원, 해외겸용 10,000 원
카드혜택: 대중교통/택시 10% 결제일할인, 편의점 7% 결제일할인(옵션), 이동통신 10% 할인
상세페이지: https://www.card-gorilla.com/card/detail/51
추천이유: 매일 이용하는 대중교통 요금을 10% 할인받을 수 있어 통학 부담을 줄여주어 추천했습니다. 라이프스타일 패키지 선택을 통해 편의점 7% 할인 혜택을 챙길 수 있습니다. 연회비가 1만 원으로 저렴하여 20대가 발급받기 좋습니다.
====================
[카드 3]
카드사: KB국민카드
카드이름: KB국민 My WE:SH 카드
연회비: 국내전용 15,000 원, 해외겸용 15,000 원
카드혜택: KB Pay 결제 시 음식점/편의점 10% 할인, 이동통신 10% 할인
상세페이지: https://www.card-gorilla.com/card/detail/2441
추천이유: 식비를 아끼기 위해 편의점을 자주 이용하는 패턴에 맞춰 편의점 10% 할인이 제공되어 추천했습니다. 앱 결제(KB Pay)를 활용하면 생활 전반에서 할인 혜택을 누리기 좋습니다.
====================
[카드 4]
카드사: 현대카드
카드이름: 현대카드ZERO Edition3(할인형)
연회비: 국내전용 15,000 원, 해외겸용 15,000 원
카드혜택: 국내외 가맹점 이용 금액 0.8% 청구 할인 (전월실적 조건 및 한도 제한 없음)
상세페이지: https://www.card-gorilla.com/card/detail/2646
추천이유:
전월실적 조건이나 한도 제한이 전혀 없어 소득이 일정하지 않은 취준생도 부담 없이 사용할 수 있어 추천했습니다. 대중교통, 편의점을 포함해 어디서든 기본 할인이 적용되어 관리가 편리합니다.
        """
    },
    {
        "User_Input": "자취하는 대학생인데 공과금이랑 편의점 할인이 많이 되는 카드를 원해요 연회비는 2만원 이하였으면 좋겠어요.",
        "Output":"""
[카드 1]
카드사: 신한카드
카드이름: 신한카드 Mr.Life
연회비: 해외겸용 15,000원
카드혜택: 월납요금(공과금) 10% 할인, 편의점 10% 할인
상세페이지: https://www.card-gorilla.com/card/detail/13
추천이유: 공과금과 편의점 할인율이 10%로 매우 높아 1인 가구의 고정 지출을 줄이는 데 효과적입니다. 연회비 또한 15,000원으로 원하시는 조건에 부합합니다.
====================
[카드 2]
카드사: 삼성카드
카드이름: 삼성 iD SELECT ALL 카드
연회비: 국내 20,000원 / 해외 20,000원
카드혜택: 아파트 관리비/통신 10% 할인, 편의점 7% 할인
상세페이지: 확인 필요
추천이유: 통신비와 관리비 등 고정 지출에서 10% 할인을 제공하여 자취생에게 유리합니다. 편의점 등 일상 영역에서도 7% 할인을 받을 수 있습니다.
====================
[카드 3]
카드사: 우리카드
카드이름: 카드의정석 SHOPPING+
연회비: 국내 10,000원 / 해외 12,000원
카드혜택: 온라인/오프라인 쇼핑 10% 청구할인
상세페이지: 확인 필요
추천이유: 자취에 필요한 물품을 온/오프라인 쇼핑으로 구매하실 때 범용적인 10% 할인을 받을 수 있는 카드입니다. 연회비 부담이 적습니다.
====================
[카드 4]
카드사: 현대카드
카드이름: 현대카드ZERO MOBILE Edition2(할인형)
연회비: 확인 필요
카드혜택: 전 가맹점 할인 (상세 확인 필요)
상세페이지: 확인 필요
추천이유: 특정 가맹점이나 실적 조건에 신경 쓰기 번거로우실 때, 전 가맹점 기본 할인을 통해 편하게 사용하실 수 있는 범용 카드입니다.
        """
    },
    {
        "User_Input": "요즘 혜택 좋은 신용카드 아무거나 4개만 추천해주세요.",
        "Output":"""
[카드 1]
카드사: 현대카드
카드이름: 현대카드ZERO Edition3(할인형)
연회비: 국내전용 15,000 원, 해외겸용 15,000 원
카드혜택: 국내외 가맹점 이용 금액 0.8% 청구 할인 (전월실적 조건 및 한도 제한 없음)
상세페이지: https://www.card-gorilla.com/card/detail/2646
추천이유: 복잡한 전월 실적 조건 없이 모든 가맹점에서 기본 할인을 받을 수 있는 범용성 높은 카드라 추천했습니다. 카드 관리가 번거로운 분들에게 최적화된 무조건 할인 유형입니다.
====================
[카드 2]
카드사: 롯데카드
카드이름: LOCA 365 카드
연회비: 국내전용 20,000 원, 해외겸용 20,000 원
카드혜택: 아파트관리비, 공과금, 이동통신, 대중교통 10% 결제일 할인
상세페이지: https://www.card-gorilla.com/card/detail/2330
추천이유: 매월 발생하는 공과금, 아파트관리비, 통신비 등 고정지출 영역에서 강력한 10% 할인을 제공하여 추천했습니다. 고정비 절감 효과가 커 직장인과 1인 가구 모두에게 인기가 많습니다.
====================
[카드 3]
카드사: 삼성카드
카드이름: 삼성카드 taptap O
연회비: 국내전용 10,000 원, 해외겸용 10,000 원
카드혜택: 스타벅스 50% 할인 또는 커피전문점 30% 할인, 쇼핑 7% 할인
상세페이지: https://www.card-gorilla.com/card/detail/51
추천이유: 커피전문점 최대 50% 할인 및 온라인 쇼핑 할인을 제공하여 2030 트렌드 소비에 맞춰 추천했습니다. 본인의 소비 패턴에 맞게 혜택 패키지를 매월 변경할 수 있는 것이 장점입니다.
====================
[카드 4]
카드사: 삼성카드
카드이름: 삼성카드 & MILEAGE PLATINUM (스카이패스)
연회비: 국내전용 47,000 원, 해외겸용 49,000 원
카드혜택: 모든 가맹점 1천원당 1 마일리지 기본 적립, 백화점/주유 등 2 마일리지 특별 적립
상세페이지: https://www.card-gorilla.com/card/detail/49
추천이유: 여행을 즐기시는 분들을 위한 대한항공 마일리지 적립 특화 카드로 추천했습니다. 전월 실적 조건 없이 결제 금액 1천 원당 마일리지가 지속적으로 적립되어 마일리지 모으기에 효율적입니다.
        """
    },
    {
        "User_Input": "쓸만한 신용카드 하나 추천해주세요.",
        "Output":"""
[카드 1]
카드사: 현대카드
카드이름: 현대카드ZERO(할인형)
연회비: 확인 필요
카드혜택: 조건 없는 가맹점 기본 할인
상세페이지: 확인 필요
추천이유: 전월 실적 조건 없이 혜택을 받을 수 있어 관리가 편한 범용 카드입니다.
====================
[카드 2]
카드사: 신한카드
카드이름: 신한카드 Mr.Life
연회비: 해외겸용 15,000원
카드혜택: 월납요금(공과금) 10% 할인, 병원/약국 10% 할인
상세페이지: https://www.card-gorilla.com/card/detail/13
추천이유: 매월 발생하는 공과금, 병원비 등 일상적인 고정 지출을 방어하는 데 특화된 카드입니다.
====================
[카드 3]
카드사: 삼성카드
카드이름: 삼성 iD SELECT ALL 카드
연회비: 국내 20,000원 / 해외 20,000원
카드혜택: 디지털콘텐츠 멤버십 50% 할인, 배달앱 7% 할인
상세페이지: 확인 필요
추천이유: OTT 등의 디지털 구독과 배달앱 혜택이 강력하여 트렌디한 소비 생활에 적합합니다.
====================
[카드 4]
카드사: 삼성카드
카드이름: 삼성 iD SELECT ALL 카드
연회비: 국내 20,000원 / 해외 20,000원
카드혜택: 해외 2% 할인
상세페이지: 확인 필요
추천이유: 해외 결제 시 2% 할인을 제공하여 여행이나 해외 직구를 즐기시는 분들께 유리합니다.
        """
    }
]


# FewShot 템플릿 생성
example_template = ChatPromptTemplate([
    ("user", "{User_Input}"),
    ("ai", "{Output}")
])

# 모델 전달 템플릿 생성
fewshot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_template
)

final_template = ChatPromptTemplate([
    ("system", system_prompt),
    fewshot_prompt,
    ("user", "{input_data}")
])


# FewShot 체인 생성
fewshot_chain = final_template | model | StrOutputParser()

print("첫번째 고객\n", input_data1)
print()
print("답변:\n", fewshot_chain.invoke(input_data1))
print()
print("두번째 고객\n", input_data2)
print()
print("답변:\n", fewshot_chain.invoke(input_data2))
print()
print("세번째 고객\n", input_data3)
print()
print("답변:\n", fewshot_chain.invoke(input_data3))
print()
print("네번째 고객\n", input_data4)
print()
print("답변:\n", fewshot_chain.invoke(input_data4))
print()
print("다섯번째 고객\n", input_data5)
print()
print("답변:\n", fewshot_chain.invoke(input_data5))

첫번째 고객
 저는 취준생이라 소득이 적어요. 매일 지하철+버스 타고 학원 다니고, 밥은 편의점에서 해결합니다. 연회비 부담 없고 대중교통이랑 편의점 할인 되는 카드 추천해주세요.

답변:
 [카드 1]
카드사: 현대카드
카드이름: 현대카드 ZERO(할인형)
연회비: 무료
카드혜택: 대중교통/편의점 할인 등 기본 할인
상세페이지: 확인 필요
추천이유: 연회비 0원으로 시작하기 좋고 학원 출퇴근 등 대중교통과 편의점 지출에 실질적 혜택이 있습니다.
[카드 2]
카드사: 신한카드
카드이름: 신한카드 Shine ZERO
연회비: 무료
카드혜택: 편의점 할인 포함 기본 할인
상세페이지: 확인 필요
추천이유: 편의점 지출을 커버하기 쉬운 무연카드로 일상비용 관리에 유리합니다.
[카드 3]
카드사: KB국민카드
카드이름: KB국민 ZERO
연회비: 무료
카드혜택: 대중교통 할인 포함 기본 할인
상세페이지: 확인 필요
추천이유: 대중교통 할인으로 학원 이동 비용 절감에 도움을 주고 연회비가 없습니다.
[카드 4]
카드사: 하나카드
카드이름: 하나카드 ZERO
연회비: 무료
카드혜택: 대중교통/편의점 등 기본 할인
상세페이지: 확인 필요
추천이유: 연회비 없이 대중교통과 편의점 지출에 혜택을 집중시켜 쓰기 편합니다.

두번째 고객
 30대 직장인입니다. 온라인 쇼핑이랑 카페를 많이 이용하고 해외여행도 자주 갑니다. 해외결제 수수료 낮고 쇼핑 할인 되는 카드 추천해주세요. 연회비 3만원까지 괜찮아요.

답변:
 [카드 1]
카드사: KB국민카드
카드이름: 카드의정석 SHOPPING+
연회비: 국내전용 10,000 원, 해외겸용 12,000 원
카드혜택: 온라인/오프라인 쇼핑 10% 청구 할인
상세페이지: https://www.card-gorilla.com/card/detail/2441
추천이유: 온라인 쇼핑에서 10% 할인으로 큰 폭의 절감이 가능합니다. 연회비가 1만 원대라 부담이 적고 해외 사용 시에도 비교적 합리적입니다. 쇼핑 중심 혜택으로 실용성이 높습니다. 해외여행 

# 3. RAG 모델

In [18]:
path = "./data/rag_chunks_with_rankings.jsonl"

docs = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        docs.append(
            Document(
                page_content=str(line)
                )
            )

splitter = CharacterTextSplitter(
    separator="", # 분할 기준
    chunk_size=600,
    chunk_overlap=100
)

embedding = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory="./Chroma",
    collection_name="cardGorilla2"
)

vectorstore = Chroma(
    embedding_function=embedding,
    persist_directory="./Chroma",
    collection_name="cardGorilla2"
)

user_prompt = """\
아래 사용자의 정보를 보고 그 상황에 맞는 알맞은 카드를 추천해주세요.

[사용자 정보]
{input_data}

[카드 정보]
{context}\
"""



retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 50,
        "fetch_k": 1000,
        "lambda_mult": 0.8
    }
)
result1 = retriever.invoke(input_data1)
result2 = retriever.invoke(input_data2)
result3 = retriever.invoke(input_data3)
result4 = retriever.invoke(input_data4)
result5 = retriever.invoke(input_data5)



rag_system_prompt = """\
<System_Prompt>
    <Role_Assignment>
        당신은 "한국 신용카드 추천 챗봇"이다. 반드시 제공된 <context> 데이터와 <user_prompt>에 포함된 정보에 기반하여, 유저에게 가장 적합한 카드 4종의 추천을 단 한 번의 응답으로 제공해야 한다.
    </Role_Assignment>

    <Absolute_Rules>
    <!-- AI instruction: Follow these internal rules STRICTLY for generating outputs. -->
    1) 질문 금지: 어떤 경우에도 사용자에게 질문/역질문/확인 요청을 하지 않는다.
    2) RAG 기반 추천: 추천 카드는 반드시 <context> 내에 존재하는 카드 중에서만 선택한다. 절대 존재하지 않는 카드를 창작해 답변하지 않는다. 카드 정보 또한 <context> 내에 존재하는 정보만을 사용한다.
        - 유저가 명시한 선호/제약(연회비 상한, 혜택 카테고리, 카드사 선호/비선호, 해외/국내, 할인/적립/마일리지, 전월실적 부담 등)이 있으면 최우선 반영한다.
        - 유저 프롬프트가 모호하거나 정보가 부족할 경우, 즉시 <Fallback_Logic>(정보 부족 모드)로 이행하고 범용 포트폴리오(4유형 커버리지)를 기반으로 추천한다.
    3) 출력 포맷 강제: 출력은 <Output_Format>에서 지정된 카드 블록과 구분선만 허용한다. 인사말, 서론, 요약, 면책 조항, 추가 안내 등의 출력을 금지한다.
    4) 카드 수: 반드시 4개만 출력한다(초과/미만 금지).
    5) 사실성 유지: <context> 내에 연회비/혜택/URL 정보가 확실하지 않을 경우, 절대 추정하지 않고 해당 항목에 "확인 필요"라고 기입한다. 카드 신규 발급 여부가 불확실할 경우, 카드 혜택에 "발급 가능 여부 확인 필요"라고 기입한다.
    6) 추천이유 규칙: 추천 이유는 5줄 이내(줄바꿈 기준)로 작성한다. 질문 및 과장된 표현(무조건, 최고, 절대 등) 사용을 금지한다.
    7) 문체 지정: 한국어 존댓말을 사용하며, 문장 종결은 “~니다.” 체를 사용한다.
    </Absolute_Rules>

    <Fallback_Logic>
    정보 부족 모드(유저 정보가 거의 주어지지 않았을 경우 내부 선택 로직)
    - 대표 소비 패턴 4유형을 커버하도록 1개씩 선택한다:
    A) 조건 단순/관리 부담 낮음(범용 할인/적립)
    B) 고정지출(통신/공과금/생활비)
    C) 트렌드/구독/외식·배달
    D) 여행/마일리지/해외결제
    - 선택 우선순위: 접근성(연회비/조건 과도X) > 범용성 > 컨셉 명확성 > 검증성
    - 단, 유저 프롬프트에 특정 카테고리/제약이 있으면 해당 축의 가중치를 가장 높게 둔다.
    </Fallback_Logic>

    <Output_Format>
    출력 형식(정확히 이 구조 + 카드 사이 구분선 필수)
    [카드 1]
    카드사: ...
    카드이름: ...
    연회비: ...
    카드혜택: ...
    상세페이지: ...
    추천이유: ...
    (추천이유는 최대 5줄)
    ====================
    [카드 2]
    카드사: ...
    카드이름: ...
    연회비: ...
    카드혜택: ...
    상세페이지: ...
    추천이유: ...
    (추천이유는 최대 5줄)
    ====================
    [카드 3]
    카드사: ...
    카드이름: ...
    연회비: ...
    카드혜택: ...
    상세페이지: ...
    추천이유: ...
    (추천이유는 최대 5줄)
    ====================
    [카드 4]
    카드사: ...
    카드이름: ...
    연회비: ...
    카드혜택: ...
    상세페이지: ...
    추천이유: ...
    (추천이유는 최대 5줄)
    </Output_Format>

    <Few_Shot_Examples>
    <Example_A type="구체적인 조건이 있는 경우">
        <User_Query_1> "저는 20대 취준생입니다. 매일 대중교통으로 학원에 가고, 식비는 주로 편의점에서 해결해요. 전월실적 부담이 적고 대중교통, 편의점 할인이 되는 카드를 추천해주세요."</User_Query>
        <AI_Response_1>
            [카드 1]
            카드사: 신한카드
            카드이름: 신한카드 Mr.Life
            연회비: 해외겸용 15,000 원
            카드혜택: 편의점 10% 할인, 병원/약국 10% 할인, 공과금 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/13
            추천이유: 편의점 결제 시 10% 할인을 제공하여 식비 절감에 유리해 추천했습니다. 공과금과 통신비 할인도 포함되어 자취하는 취업준비생의 고정지출 방어에 적합합니다. 전월실적 30만 원으로 비교적 유지 부담이 적은 편입니다.
            ====================
            [카드 2]
            카드사: 삼성카드
            카드이름: 삼성카드 taptap O
            연회비: 국내전용 10,000 원, 해외겸용 10,000 원
            카드혜택: 대중교통/택시 10% 결제일할인, 편의점 7% 결제일할인(옵션), 이동통신 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/51
            추천이유: 매일 이용하는 대중교통 요금을 10% 할인받을 수 있어 통학 부담을 줄여주어 추천했습니다. 라이프스타일 패키지 선택을 통해 편의점 7% 할인 혜택을 챙길 수 있습니다. 연회비가 1만 원으로 저렴하여 20대가 발급받기 좋습니다.
            ====================
            [카드 3]
            카드사: KB국민카드
            카드이름: KB국민 My WE:SH 카드
            연회비: 국내전용 15,000 원, 해외겸용 15,000 원
            카드혜택: KB Pay 결제 시 음식점/편의점 10% 할인, 이동통신 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/2441
            추천이유: 식비를 아끼기 위해 편의점을 자주 이용하는 패턴에 맞춰 편의점 10% 할인이 제공되어 추천했습니다. 앱 결제(KB Pay)를 활용하면 생활 전반에서 할인 혜택을 누리기 좋습니다.
            ====================
            [카드 4]
            카드사: 현대카드
            카드이름: 현대카드ZERO Edition3(할인형)
            연회비: 국내전용 15,000 원, 해외겸용 15,000 원
            카드혜택: 국내외 가맹점 이용 금액 0.8% 청구 할인 (전월실적 조건 및 한도 제한 없음)
            상세페이지: https://www.card-gorilla.com/card/detail/2646
            추천이유:
            전월실적 조건이나 한도 제한이 전혀 없어 소득이 일정하지 않은 취준생도 부담 없이 사용할 수 있어 추천했습니다. 대중교통, 편의점을 포함해 어디서든 기본 할인이 적용되어 관리가 편리합니다.
        </AI_Response_1>

        <User_Query_2> "자취하는 대학생인데 공과금이랑 편의점 할인이 많이 되는 카드를 원해요. 연회비는 2만원 이하였으면 좋겠어요."</User_Query_2> 
        <AI_Response_2>
            [카드 1]
            카드사: 신한카드
            카드이름: 신한카드 Mr.Life
            연회비: 해외겸용 15,000원
            카드혜택: 월납요금(공과금) 10% 할인, 편의점 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/13
            추천이유: 공과금과 편의점 할인율이 10%로 매우 높아 1인 가구의 고정 지출을 줄이는 데 효과적입니다. 연회비 또한 15,000원으로 원하시는 조건에 부합합니다.
            ====================
            [카드 2]
            카드사: 삼성카드
            카드이름: 삼성 iD SELECT ALL 카드
            연회비: 국내 20,000원 / 해외 20,000원
            카드혜택: 아파트 관리비/통신 10% 할인, 편의점 7% 할인
            상세페이지: 확인 필요
            추천이유: 통신비와 관리비 등 고정 지출에서 10% 할인을 제공하여 자취생에게 유리합니다. 편의점 등 일상 영역에서도 7% 할인을 받을 수 있습니다.
            ====================
            [카드 3]
            카드사: 우리카드
            카드이름: 카드의정석 SHOPPING+
            연회비: 국내 10,000원 / 해외 12,000원
            카드혜택: 온라인/오프라인 쇼핑 10% 청구할인
            상세페이지: 확인 필요
            추천이유: 자취에 필요한 물품을 온/오프라인 쇼핑으로 구매하실 때 범용적인 10% 할인을 받을 수 있는 카드입니다. 연회비 부담이 적습니다.
            ====================
            [카드 4]
            카드사: 현대카드
            카드이름: 현대카드ZERO MOBILE Edition2(할인형)
            연회비: 확인 필요
            카드혜택: 전 가맹점 할인 (상세 확인 필요)
            상세페이지: 확인 필요
            추천이유: 특정 가맹점이나 실적 조건에 신경 쓰기 번거로우실 때, 전 가맹점 기본 할인을 통해 편하게 사용하실 수 있는 범용 카드입니다.
        </AI_Response_2>
    </Example_A>
    
    <Example_B type="정보 부족 모드가 발동되는 경우">
        <User_Query_1> "요즘 혜택 좋은 신용카드 아무거나 4개만 추천해주세요."</User_Query_2>
        <AI_Response_1>
            [카드 1]
            카드사: 현대카드
            카드이름: 현대카드ZERO Edition3(할인형)
            연회비: 국내전용 15,000 원, 해외겸용 15,000 원
            카드혜택: 국내외 가맹점 이용 금액 0.8% 청구 할인 (전월실적 조건 및 한도 제한 없음)
            상세페이지: https://www.card-gorilla.com/card/detail/2646
            추천이유: 복잡한 전월 실적 조건 없이 모든 가맹점에서 기본 할인을 받을 수 있는 범용성 높은 카드라 추천했습니다. 카드 관리가 번거로운 분들에게 최적화된 무조건 할인 유형입니다.
            ====================
            [카드 2]
            카드사: 롯데카드
            카드이름: LOCA 365 카드
            연회비: 국내전용 20,000 원, 해외겸용 20,000 원
            카드혜택: 아파트관리비, 공과금, 이동통신, 대중교통 10% 결제일 할인
            상세페이지: https://www.card-gorilla.com/card/detail/2330
            추천이유: 매월 발생하는 공과금, 아파트관리비, 통신비 등 고정지출 영역에서 강력한 10% 할인을 제공하여 추천했습니다. 고정비 절감 효과가 커 직장인과 1인 가구 모두에게 인기가 많습니다.
            ====================
            [카드 3]
            카드사: 삼성카드
            카드이름: 삼성카드 taptap O
            연회비: 국내전용 10,000 원, 해외겸용 10,000 원
            카드혜택: 스타벅스 50% 할인 또는 커피전문점 30% 할인, 쇼핑 7% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/51
            추천이유: 커피전문점 최대 50% 할인 및 온라인 쇼핑 할인을 제공하여 2030 트렌드 소비에 맞춰 추천했습니다. 본인의 소비 패턴에 맞게 혜택 패키지를 매월 변경할 수 있는 것이 장점입니다.
            ====================
            [카드 4]
            카드사: 삼성카드
            카드이름: 삼성카드 & MILEAGE PLATINUM (스카이패스)
            연회비: 국내전용 47,000 원, 해외겸용 49,000 원
            카드혜택: 모든 가맹점 1천원당 1 마일리지 기본 적립, 백화점/주유 등 2 마일리지 특별 적립
            상세페이지: https://www.card-gorilla.com/card/detail/49
            추천이유: 여행을 즐기시는 분들을 위한 대한항공 마일리지 적립 특화 카드로 추천했습니다. 전월 실적 조건 없이 결제 금액 1천 원당 마일리지가 지속적으로 적립되어 마일리지 모으기에 효율적입니다.
        </AI_Response_1>
 
        <User_Query_2>"요즘 쓸만한 신용카드 하나 추천해주세요."</User_Query_2>
        <AI_Response_2>
            [카드 1]
            카드사: 현대카드
            카드이름: 현대카드ZERO(할인형)
            연회비: 확인 필요
            카드혜택: 조건 없는 가맹점 기본 할인
            상세페이지: 확인 필요
            추천이유: 전월 실적 조건 없이 혜택을 받을 수 있어 관리가 편한 범용 카드입니다.
            ====================
            [카드 2]
            카드사: 신한카드
            카드이름: 신한카드 Mr.Life
            연회비: 해외겸용 15,000원
            카드혜택: 월납요금(공과금) 10% 할인, 병원/약국 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/13
            추천이유: 매월 발생하는 공과금, 병원비 등 일상적인 고정 지출을 방어하는 데 특화된 카드입니다.
            ====================
            [카드 3]
            카드사: 삼성카드
            카드이름: 삼성 iD SELECT ALL 카드
            연회비: 국내 20,000원 / 해외 20,000원
            카드혜택: 디지털콘텐츠 멤버십 50% 할인, 배달앱 7% 할인
            상세페이지: 확인 필요
            추천이유: OTT 등의 디지털 구독과 배달앱 혜택이 강력하여 트렌디한 소비 생활에 적합합니다.
            ====================
            [카드 4]
            카드사: 삼성카드
            카드이름: 삼성 iD SELECT ALL 카드
            연회비: 국내 20,000원 / 해외 20,000원
            카드혜택: 해외 2% 할인
            상세페이지: 확인 필요
            추천이유: 해외 결제 시 2% 할인을 제공하여 여행이나 해외 직구를 즐기시는 분들께 유리합니다.
        </AI_Resopnse_2>
    </Example_B>
    </Few_Shot_Examples>
</System_Prompt>\
"""

rag_template = ChatPromptTemplate([
    ("system", rag_system_prompt),
    ("user", user_prompt)
])

rag_chain = rag_template | model | StrOutputParser()

print("첫번째 고객\n", input_data1)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data1, "context": result1}))
print()
print("두번째 고객\n", input_data2)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data2, "context": result2}))
print()
print("세번째 고객\n", input_data3)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data3, "context": result3}))
print()
print("네번째 고객\n", input_data4)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data4, "context": result4}))
print()
print("다섯번째 고객\n", input_data5)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data5, "context": result5}))

첫번째 고객
 저는 취준생이라 소득이 적어요. 매일 지하철+버스 타고 학원 다니고, 밥은 편의점에서 해결합니다. 연회비 부담 없고 대중교통이랑 편의점 할인 되는 카드 추천해주세요.

답변:
 [카드 1]
카드사: IBK기업은행
카드이름: K-패스 (신용)
연회비: 국내 2,000원 / 해외 4,000원
카드혜택: 대중교통 5% 할인
상세페이지: https://www.card-gorilla.com/card/detail/2559
추천이유: 전월실적 조건이 200,000원 이상으로 비교적 낮은 편입니다. 대중교통 할인율이 5%로 실질적인 비용 절감이 큽니다. 연회비가 매우 저렴해 취준생의 초기 비용 부담이 작습니다. 지하철/버스 사용이 많은 상황에 적합합니다. 해외 이용도 비교적 저렴한 편입니다.


[카드 2]
카드사: BC 바로카드
카드이름: BC 바로 On&Off 카드
연회비: 국내 5,000원 / 해외 5,000원
카드혜택: 대중교통 10% 청구할인
상세페이지: https://www.card-gorilla.com/card/detail/2591
추천이유: 대중교통 10% 할인으로 교통비를 크게 절감할 수 있습니다. 연회비가 5,000원으로 매우 저렴합니다. 전월실적 조건이 300,000원 이상으로 부담이 비교적 작습니다. 버스/지하철/택시 모두 할인 대상이므로 활용도가 높습니다. 편의점 할인 혜택은 별도 조건 확인이 필요합니다.


[카드 3]
카드사: 신한카드
카드이름: K-패스 신한카드
연회비: 국내 7,000원 / 해외 10,000원
카드혜택: 대중교통 10% 할인
상세페이지: https://www.card-gorilla.com/card/detail/2690
추천이유: 대중교통 10% 할인으로 교통비를 효과적으로 줄여줍니다. 연회비가 비교적 합리적이며 해외 이용 시 비용도 비교적 낮습니다. 전월실적 300,000원 이상이 필요합니다. 지하철/버스 이용에 유용한 기본 혜택이 있습니다.


[카드 4]
카드사: 삼성카드
카드이름: 네이버페이 taptap (삼

In [16]:
path1 = "./data/rag_chunks.jsonl"
path2 = "./data/card_ranking_for_rag.jsonl"

docs = []
with open(path1, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        docs.append(
            Document(
                page_content=str(line)
                )
            )

with open(path2, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        docs.append(
            Document(
                page_content=str(line)
                )
            )

splitter = CharacterTextSplitter(
    separator="", # 분할 기준
    chunk_size=600,
    chunk_overlap=100
)

embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# vectorstore = Chroma.from_documents(
#     documents=docs,
#     embedding=embedding,
#     persist_directory="./Chroma",
#     collection_name="cardGorilla"
# )

vectorstore = Chroma(
    embedding_function=embedding,
    persist_directory="./Chroma",
    collection_name="cardGorilla"
)

user_prompt = """\
아래 사용자의 정보를 보고 그 상황에 맞는 알맞은 카드를 추천해주세요.

[사용자 정보]
{input_data}

[카드 정보]
{context}\
"""



retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 50,
        "fetch_k": 1000,
        "lambda_mult": 0.8
    }
)
result1 = retriever.invoke(input_data1)
result2 = retriever.invoke(input_data2)
result3 = retriever.invoke(input_data3)
result4 = retriever.invoke(input_data4)
result5 = retriever.invoke(input_data5)



rag_system_prompt = """\
<System_Prompt>
    <Role_Assignment>
        당신은 "한국 신용카드 추천 챗봇"이다. 반드시 제공된 <context> 데이터와 <user_prompt>에 포함된 정보에 기반하여, 유저에게 가장 적합한 카드 4종의 추천을 단 한 번의 응답으로 제공해야 한다.
    </Role_Assignment>

    <Absolute_Rules>
    <!-- AI instruction: Follow these internal rules STRICTLY for generating outputs. -->
    1) 질문 금지: 어떤 경우에도 사용자에게 질문/역질문/확인 요청을 하지 않는다.
    2) RAG 기반 추천: 추천 카드는 반드시 <context> 내에 존재하는 카드 중에서만 선택한다. 절대 존재하지 않는 카드를 창작해 답변하지 않는다. 카드 정보 또한 <context> 내에 존재하는 정보만을 사용한다.
        - 유저가 명시한 선호/제약(연회비 상한, 혜택 카테고리, 카드사 선호/비선호, 해외/국내, 할인/적립/마일리지, 전월실적 부담 등)이 있으면 최우선 반영한다.
    3) 출력 포맷 강제: 출력은 <Output_Format>에서 지정된 카드 블록과 구분선만 허용한다. 인사말, 서론, 요약, 면책 조항, 추가 안내 등의 출력을 금지한다.
    4) 카드 수: 반드시 4개만 출력한다(초과/미만 금지).
    5) 사실성 유지: <context> 내에 한 카드의 이름을 찾았다면 해당 카드의 모든 혜택을 끝까지 검색하여 통합한다. 한 카드의 모든 카드 혜택을 가져와 출력한다.
    6) 추천이유 규칙: 추천 이유는 5줄 이내(줄바꿈 기준)로 작성한다. 질문 및 과장된 표현(무조건, 최고, 절대 등) 사용을 금지한다.
    7) 문체 지정: 한국어 존댓말을 사용하며, 문장 종결은 “~니다.” 체를 사용한다.
    </Absolute_Rules>

    <Output_Format>
    출력 형식(정확히 이 구조 + 카드 사이 구분선 필수)
    [카드 1]
    카드사: ...
    카드이름: ...
    연회비: ...
    카드혜택: ...
    상세페이지: ...
    추천이유: ...
    (추천이유는 최대 5가지)
    ====================
    [카드 2]
    카드사: ...
    카드이름: ...
    연회비: ...
    카드혜택: ...
    상세페이지: ...
    추천이유: ...
    (추천이유는 최대 5줄)
    ====================
    [카드 3]
    카드사: ...
    카드이름: ...
    연회비: ...
    카드혜택: ...
    상세페이지: ...
    추천이유: ...
    (추천이유는 최대 5줄)
    ====================
    [카드 4]
    카드사: ...
    카드이름: ...
    연회비: ...
    카드혜택: ...
    상세페이지: ...
    추천이유: ...
    (추천이유는 최대 5줄)
    </Output_Format>

    <Few_Shot_Examples>
    <Example_A type="구체적인 조건이 있는 경우">
        <User_Query_1> "저는 20대 취준생입니다. 매일 대중교통으로 학원에 가고, 식비는 주로 편의점에서 해결해요. 전월실적 부담이 적고 대중교통, 편의점 할인이 되는 카드를 추천해주세요."</User_Query>
        <AI_Response_1>
            [카드 1]
            카드사: 신한카드
            카드이름: 신한카드 Mr.Life
            연회비: 해외겸용 15,000 원
            카드혜택: 편의점 10% 할인, 병원/약국 10% 할인, 공과금 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/13
            추천이유: 편의점 결제 시 10% 할인을 제공하여 식비 절감에 유리해 추천했습니다. 공과금과 통신비 할인도 포함되어 자취하는 취업준비생의 고정지출 방어에 적합합니다. 전월실적 30만 원으로 비교적 유지 부담이 적은 편입니다.
            ====================
            [카드 2]
            카드사: 삼성카드
            카드이름: 삼성카드 taptap O
            연회비: 국내전용 10,000 원, 해외겸용 10,000 원
            카드혜택: 대중교통/택시 10% 결제일할인, 편의점 7% 결제일할인(옵션), 이동통신 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/51
            추천이유: 매일 이용하는 대중교통 요금을 10% 할인받을 수 있어 통학 부담을 줄여주어 추천했습니다. 라이프스타일 패키지 선택을 통해 편의점 7% 할인 혜택을 챙길 수 있습니다. 연회비가 1만 원으로 저렴하여 20대가 발급받기 좋습니다.
            ====================
            [카드 3]
            카드사: KB국민카드
            카드이름: KB국민 My WE:SH 카드
            연회비: 국내전용 15,000 원, 해외겸용 15,000 원
            카드혜택: KB Pay 결제 시 음식점/편의점 10% 할인, 이동통신 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/2441
            추천이유: 식비를 아끼기 위해 편의점을 자주 이용하는 패턴에 맞춰 편의점 10% 할인이 제공되어 추천했습니다. 앱 결제(KB Pay)를 활용하면 생활 전반에서 할인 혜택을 누리기 좋습니다.
            ====================
            [카드 4]
            카드사: 현대카드
            카드이름: 현대카드ZERO Edition3(할인형)
            연회비: 국내전용 15,000 원, 해외겸용 15,000 원
            카드혜택: 국내외 가맹점 이용 금액 0.8% 청구 할인 (전월실적 조건 및 한도 제한 없음)
            상세페이지: https://www.card-gorilla.com/card/detail/2646
            추천이유:
            전월실적 조건이나 한도 제한이 전혀 없어 소득이 일정하지 않은 취준생도 부담 없이 사용할 수 있어 추천했습니다. 대중교통, 편의점을 포함해 어디서든 기본 할인이 적용되어 관리가 편리합니다.
        </AI_Response_1>

        <User_Query_2> "자취하는 대학생인데 공과금이랑 편의점 할인이 많이 되는 카드를 원해요. 연회비는 2만원 이하였으면 좋겠어요."</User_Query_2> 
        <AI_Response_2>
            [카드 1]
            카드사: 신한카드
            카드이름: 신한카드 Mr.Life
            연회비: 해외겸용 15,000원
            카드혜택: 월납요금(공과금) 10% 할인, 편의점 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/13
            추천이유: 공과금과 편의점 할인율이 10%로 매우 높아 1인 가구의 고정 지출을 줄이는 데 효과적입니다. 연회비 또한 15,000원으로 원하시는 조건에 부합합니다.
            ====================
            [카드 2]
            카드사: 삼성카드
            카드이름: 삼성 iD SELECT ALL 카드
            연회비: 국내 20,000원 / 해외 20,000원
            카드혜택: 아파트 관리비/통신 10% 할인, 편의점 7% 할인
            상세페이지: 실제 데이터가 들어간 완벽한 답변
            추천이유: 통신비와 관리비 등 고정 지출에서 10% 할인을 제공하여 자취생에게 유리합니다. 편의점 등 일상 영역에서도 7% 할인을 받을 수 있습니다.
            ====================
            [카드 3]
            카드사: 우리카드
            카드이름: 카드의정석 SHOPPING+
            연회비: 국내 10,000원 / 해외 12,000원
            카드혜택: 온라인/오프라인 쇼핑 10% 청구할인
            상세페이지: 실제 데이터가 들어간 완벽한 답변
            추천이유: 자취에 필요한 물품을 온/오프라인 쇼핑으로 구매하실 때 범용적인 10% 할인을 받을 수 있는 카드입니다. 연회비 부담이 적습니다.
            ====================
            [카드 4]
            카드사: 현대카드
            카드이름: 현대카드ZERO MOBILE Edition2(할인형)
            연회비: 실제 데이터가 들어간 완벽한 답변
            카드혜택: 전 가맹점 할인 (실제 데이터가 들어간 완벽한 답변)
            상세페이지: 실제 데이터가 들어간 완벽한 답변
            추천이유: 특정 가맹점이나 실적 조건에 신경 쓰기 번거로우실 때, 전 가맹점 기본 할인을 통해 편하게 사용하실 수 있는 범용 카드입니다.
        </AI_Response_2>
    </Example_A>
    </Few_Shot_Examples>
</System_Prompt>\
"""

rag_template = ChatPromptTemplate([
    ("system", rag_system_prompt),
    ("user", user_prompt)
])

rag_chain = rag_template | model | StrOutputParser()

print("첫번째 고객\n", input_data4)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data4, "context": result4}))

첫번째 고객
 맞벌이 신혼부부입니다. 주유비, 마트 장보기, 공과금 할인이 많이 되는 카드를 찾고 있어요. 연회비는 5만원까지 괜찮습니다. 롯데카드는 빼주세요.

답변:
 [카드 1]
카드사: KB국민카드
카드이름: 굿데이올림카드
연회비: 국내 15,000원 / 해외 20,000원
카드혜택: 주유소(리터당 60원 청구할인), 대형마트 10% 청구할인
상세페이지: https://www.card-gorilla.com/card/detail/115
추천이유: 주유소에서의 할인으로 주유비 부담을 크게 줄여줍니다. 대형마트 할인으로 식료품 비용도 절감됩니다. 연회비가 1.5만 원대로 부담이 작고, 전월실적 30만 원 조건이 비교적 달성하기 쉽습니다. 전반적으로 맞벌이 신혼부부의 주유/마트 지출 구조에 잘 맞습니다.

[카드 2]
카드사: 신한카드
카드이름: 신한카드 Mr.Life
연회비: 해외겸용 15,000원
카드혜택: 3대 마트 10% 할인
상세페이지: https://www.card-gorilla.com/card/detail/13
추천이유: 마트 지출을 크게 줄여줄 수 있는 강력한 할인 혜택이 있습니다. 연회비 15,000원으로 부담이 작습니다. 전월실적 30만 원 조건도 달성하기 비교적 쉽습니다. 맞벌이 부부의 주거래마트 지출에 효과적입니다.

[카드 3]
카드사: 우리카드
카드이름: 카드의정석 SHOPPING+
연회비: 국내 10,000원 / 해외 12,000원
카드혜택: 주유소(주말) 리터당 60원 청구할인
상세페이지: https://www.card-gorilla.com/card/detail/2687
추천이유: 주말 주유 할인으로 주유비를 효과적으로 절감할 수 있습니다. 연회비가 저렴하고 전월실적 30만 원 조건도 비교적 가볍습니다. 주유 외 생활 밀착 할인 혜택도 함께 기대할 수 있습니다. 가성비가 높은 기본 혜택 구성입니다.

[카드 4]
카드사: NH농협카드
카드이름: zgm.play카드
연회비: 국내 12,000원 / 해외 12,000원
카드혜택: 공과금

# 영문 프롬프트

In [19]:
path = "./data/rag_chunks_with_rankings.jsonl"

docs = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        docs.append(
            Document(
                page_content=str(line)
                )
            )

splitter = CharacterTextSplitter(
    separator="", # 분할 기준
    chunk_size=600,
    chunk_overlap=100
)

embedding = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory="./Chroma",
    collection_name="cardGorilla2"
)

vectorstore = Chroma(
    embedding_function=embedding,
    persist_directory="./Chroma",
    collection_name="cardGorilla2"
)

user_prompt = """\
아래 사용자의 정보를 보고 그 상황에 맞는 알맞은 카드를 추천해주세요.

[사용자 정보]
{input_data}

[카드 정보]
{context}\
"""



retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 50,
        "fetch_k": 1000,
        "lambda_mult": 0.8
    }
)
result1 = retriever.invoke(input_data1)
result2 = retriever.invoke(input_data2)
result3 = retriever.invoke(input_data3)
result4 = retriever.invoke(input_data4)
result5 = retriever.invoke(input_data5)



rag_system_prompt = """\
<System_Prompt>
    <Role_Assignment>
        You are a "Korean Credit Card Recommendation Chatbot". You MUST provide exactly 4 suitable card recommendations to the user in a single response, strictly based on the provided <context> data and the information in the <user_prompt>.
    </Role_Assignment>

    <Core_Directives>
    1. No Questions: DO NOT ask questions, counter-questions, or request clarification from the user under any circumstances.
    2. RAG-Based Recommendation: ONLY select cards that exist within the <context>. NEVER hallucinate or invent non-existent cards. Use ONLY the information provided in the <context>.
    3. Output ALL Benefits: You MUST extract and list ALL the card benefits mentioned in the <context>. Do not summarize them into just one benefit.
    4. Internal CoT (Chain of Thought): Before generating the final card recommendations, you MUST process the user request internally using the <Thinking_Process> block.
        - Step 1 (Keyword Extraction): Extract keywords from the user's input (e.g., Preferred/Excluded card companies, Annual fee limits, Previous month performance constraints, Target benefit categories).
        - Step 2 (Filtering): Filter the cards in the <context> based on the extracted keywords.
        - Step 3 (Selection): Select the top 4 most appropriate cards that match the criteria.
    5. Strict Output Format: You MUST exactly follow the <Output_Format> structure. Output the <Thinking_Process> block first, followed by the card blocks and separators. DO NOT output greetings, introductions, summaries, or disclaimers.
    6. Exact Card Count: You MUST output exactly 4 cards. Neither more nor less.
    7. Recommendation Reason Rules:
        - Provide the recommendation reasons as a numbered list (1, 2, 3...) with a MAXIMUM of 5 items.
        - You MUST utilize the card's ranking data from the <context> and include it as one of the reasons (e.g., "이 카드는 카드고릴라 랭킹 X위로 검증된 인기 카드입니다.").
        - DO NOT use exaggerated expressions (e.g., "무조건", "최고", "절대").
    8. Factual Accuracy: If the annual fee, previous month performance (전월실적), benefits, or URL in the <context> is unclear or missing, strictly write "확인 필요" for that specific field.
    9. Tone and Language: Strictly use polite Korean, ending sentences with the "~니다" form. DO NOT USE English.
    </Core_Directives>

    <Output_Format>
    <Thinking_Process>
    - 사용자 키워드 추출: [선호 카드사: ...], [전월실적 조건: ...], [연회비 조건: ...], [주요 혜택: ...] 등
    - <context> 필터링 및 추천: [선정된 4가지 카드 이름과 매칭 이유 간략히]
    </Thinking_Process>
    [카드 1]
    카드사: ...
    카드이름: ...
    연회비: ...
    전월실적: ...
    카드혜택: ... (컨텍스트 내 모든 혜택 나열)
    상세페이지: ...
    추천이유:
    1. ... (반드시 랭킹 정보 포함)
    2. ...
    3. ...
    4. ...
    5. ...
    (최대 5가지 항목으로 작성)
    ====================
    [카드 2]
    ... (위와 동일 구조) ...
    ====================
    [카드 3]
    ... (위와 동일 구조) ...
    ====================
    [카드 4]
    ... (위와 동일 구조) ...
    </Output_Format>

    <Few_Shot_Examples>
    <Example>
        <User_Query> "20대 취준생입니다. 매일 대중교통으로 학원에 가고 편의점을 자주 가요. 전월실적이 30만원 이하로 낮았으면 좋겠고, 신한카드나 삼성카드를 선호합니다."</User_Query>
        <AI_Response>
            <Thinking_Process>
            - 사용자 키워드 추출: [주요 혜택: 대중교통, 편의점], [전월실적 조건: 30만원 이하], [선호 카드사: 신한카드, 삼성카드]
            - 컨텍스트 필터링 및 카드 선정:
              1. 신한카드 Mr.Life: 편의점 혜택 존재, 실적 30만원, 신한카드 충족 (랭킹 1위)
              2. 삼성카드 taptap O: 대중교통/편의점 혜택 존재, 실적 30만원, 삼성카드 충족 (랭킹 5위)
              3. 신한카드 Deep Dream: 전월실적 조건 없음, 편의점 적립, 신한카드 충족 (랭킹 12위)
              4. 삼성 iD SELECT ALL 카드: 편의점 혜택, 삼성카드 충족 (랭킹 2위)
            </Thinking_Process>

            [카드 1]
            카드사: 신한카드
            카드이름: 신한카드 Mr.Life
            연회비: 해외겸용 15,000 원
            전월실적: 30만 원 이상
            카드혜택: 편의점 10% 할인, 병원/약국 10% 할인, 세탁소 10% 할인, 오후 9시~오전 9시 온라인쇼핑 10% 할인, 전기/도시가스/통신요금 10% 할인
            상세페이지: https://www.card-gorilla.com/card/detail/13
            추천이유:
            1. 2026년 2월 기준 카드고릴라 랭킹 1위를 차지할 만큼 많은 사람에게 검증된 인기 카드입니다.
            2. 편의점 결제 시 10% 할인을 제공하여 취업준비생의 식비 절감에 매우 유리합니다.
            3. 전월실적 조건이 30만 원으로 설정되어 있어 유지 부담이 적습니다.
            4. 통신요금 및 공과금 10% 할인이 포함되어 자취 고정비 방어에 탁월합니다.
            5. 사용자가 선호하는 신한카드사의 상품입니다.
            ====================
            [카드 2]
            카드사: 삼성카드
            카드이름: 삼성카드 taptap O
            연회비: 국내전용 10,000 원, 해외겸용 10,000 원
            전월실적: 30만 원 이상
            카드혜택: 대중교통/택시 10% 결제일할인, 스타벅스 50% 할인, 편의점 7% 결제일할인(옵션), 이동통신 10% 할인, CGV/롯데시네마 5,000원 할인
            상세페이지: https://www.card-gorilla.com/card/detail/51
            추천이유:
            1. 2026년 2월 기준 카드고릴라 랭킹 6위에 올라 있는 선호도 최상위 카드입니다.
            2. 매일 이용하시는 대중교통 요금을 10% 할인받을 수 있어 통학 부담을 크게 덜어줍니다.
            3. 라이프스타일 패키지 선택을 통해 편의점 7% 할인 혜택을 챙기실 수 있습니다.
            4. 연회비가 1만 원으로 저렴하고 전월실적이 30만 원이라 관리가 수월합니다.
            5. 사용자가 선호하는 삼성카드사의 상품입니다.
            ====================
            [카드 3]
            카드사: 신한카드
            카드이름: 신한카드 처음(ANNIVERSE)
            연회비: 국내전용 15,000 원, 해외겸용 18,000 원
            전월실적: 전월실적 30만 원 이상
            카드혜택: 음식점·카페·편의점·온라인쇼핑 5% 마이신한포인트 적립, 생활·여행·패션 5% 마이신한포인트적립, 통신 10% , OTT 15%, 멤버십 20% 마이신한포인트 적립, 소비관리 보너스(계획소비 적립, 즉시결제 적립)
            상세페이지: 확인 필요
            추천이유:
            1. 2026년 2월 기준 카드고릴라 랭킹 12위에 위치한 대표적인 적립형 카드입니다.
            3. 편의점, 온라인쇼핑 등 20대 대학생이 자주 소비하는 분야에서 5% 포인트 적립을 제공합니다.
            4. 소비관리 보너스(계획소비 적립, 즉시결제 적립) 혜택이 있어 범용성이 뛰어납니다.
            5. 사용자가 명시한 신한카드사의 상품입니다.
            ====================
            [카드 4]
            카드사: 삼성카드
            카드이름: 삼성 iD SELECT ALL 카드
            연회비: 국내 20,000 원, 해외 20,000 원
            전월실적: 확인 필요
            카드혜택: 아파트 관리비/통신 10% 할인, 편의점 7% 할인, 배달앱 7% 할인, 디지털콘텐츠 멤버십 50% 할인
            상세페이지: 확인 필요
            추천이유:
            1. 2026년 2월 기준 카드고릴라 랭킹 2위를 차지한 대세 카드입니다.
            2. 편의점 7% 할인을 비롯해 배달앱 할인까지 제공되어 식비 관리에 유리합니다.
            3. OTT 등 디지털 구독 50% 할인이 포함되어 있어 20대 라이프스타일에 부합합니다.
            4. 통신비 10% 할인을 통해 추가적인 고정지출 방어가 가능합니다.
            5. 선호하시는 삼성카드 라인업 중 최근 가장 트렌디한 혜택을 담고 있습니다.
        </AI_Response>
    </Example>
    </Few_Shot_Examples>
</System_Prompt>
"""

rag_template = ChatPromptTemplate([
    ("system", rag_system_prompt),
    ("user", user_prompt)
])

rag_chain = rag_template | model | StrOutputParser()

print("첫번째 고객\n", input_data1)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data1, "context": result1}))
print()
print("두번째 고객\n", input_data2)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data2, "context": result2}))
print()
print("세번째 고객\n", input_data3)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data3, "context": result3}))
print()
print("네번째 고객\n", input_data4)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data4, "context": result4}))
print()
print("다섯번째 고객\n", input_data5)
print()
print("답변:\n", rag_chain.invoke({"input_data": input_data5, "context": result5}))

첫번째 고객
 저는 취준생이라 소득이 적어요. 매일 지하철+버스 타고 학원 다니고, 밥은 편의점에서 해결합니다. 연회비 부담 없고 대중교통이랑 편의점 할인 되는 카드 추천해주세요.

답변:
 <Thinking_Process>
- 사용자 키워드 추출: [선호 혜택: 대중교통 할인], [편의점 할인 가능 여부: 원함], [연회비 조건: 부담 없는 수준], [전월실적 조건: 낮은 편이 좋음]
- 컨텍스트 필터링 및 추천: 전월실적 조건이 낮고 연회비가 저렴하며 대중교통 할인 혜택이 확실한 카드를 우선 선택
  1) 굿데이카드 (KB국민카드) - 대중교통 10% 청구할인, 전월실적 300,000원 이상, 연회비 5,000원 국내/10,000원 해외, 랭킹 24위
  2) K-패스 신한카드 - 대중교통(버스·지하철) 10% 할인가결일 할인, 전월실적 300,000원 이상, 연회비 국내 7,000원 / 해외 10,000원, 랭킹 97위
  3) 다담카드 (KB국민카드) - 버스/지하철 10% 청구할인, 전월실적 300,000원 이상, 연회비 해외겸용 15,000원, 랭킹 84위
  4) BC 바로 클리어 플러스 - 대중교통 7% 할인, 전월실적 150,000원 이상, 연회비 국내/해외 5,000원씩, 랭킹 88위
</Thinking_Process>

[카드 1]
카드사: KB국민카드
카드이름: 굿데이카드
연회비: 국내 5,000원 / 해외 10,000원
전월실적: 300,000원 이상
카드혜택: 버스, 지하철, 택시 업종 10% 청구할인
상세페이지: https://www.card-gorilla.com/card/detail/106
추천이유:
1. 2026년 2월 기준 카드고릴라 랭킹 24위로 비교적 안정적인 인기 카드입니다.
2. 대중교통 할인 폭이 10%로 크고 자주 이용하시는 학원 통근에 유리합니다.
3. 전월실적 조건이 30만원으로 비교적 낮아 취준생에게도 관리가 용이합니다.
4. 연회비가 저렴하며 국내·해외 이용 시 비용 차이가 작습니다.
5. 대중교통 혜택이 명확